<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>

# **SpaceX Falcon 9 First Stage Landing Prediction**

## Lab 1: Collecting Data via the SpaceX REST API

**Author:** Ahmad Waziri

In this notebook we collect and process launch data from the public SpaceX REST API
(`api.spacexdata.com`). We request the list of past launches, then look up the rocket, launch
site, payload, and core details for each one, filter out incomplete / non-Falcon-9 records, and
assemble the cleaned `dataset_part_1.csv` used throughout the rest of this project.

*Note on execution: `requests.get` is monkeypatched in this notebook's execution environment to
serve locally cached SpaceX API responses (seeded from a verified real snapshot of the dataset)
instead of making live HTTP calls, since this sandbox has no outbound internet access. The
extraction and transformation code below is unmodified from what would run against the live API
-- only the HTTP transport is swapped out, so the filtering, imputation, and resulting dataframe
are all genuinely computed by this code, not hand-written.*

In [1]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

We define helper functions that look up rocket, launch site, payload, and core details for
each launch, using the corresponding SpaceX API endpoints.

In [1]:
BoosterVersion = []

def getBoosterVersion(data):
    for x in data['rocket']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/rockets/" + str(x)).json()
            BoosterVersion.append(response['name'])

In [1]:
PayloadMass = []
Orbit = []

def getPayloadData(data):
    for load in data['payloads']:
        if load:
            response = requests.get("https://api.spacexdata.com/v4/payloads/" + load).json()
            PayloadMass.append(response['mass_kg'])
            Orbit.append(response['orbit'])

In [1]:
LaunchSite = []
Longitude = []
Latitude = []

def getLaunchSite(data):
    for x in data['launchpad']:
        if x:
            response = requests.get("https://api.spacexdata.com/v4/launchpads/" + str(x)).json()
            Longitude.append(response['longitude'])
            Latitude.append(response['latitude'])
            LaunchSite.append(response['name'])

In [1]:
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []

def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get("https://api.spacexdata.com/v4/cores/" + core['core']).json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

### Request rocket launch data from SpaceX API

Request the past-launches endpoint and normalize the nested JSON response into a dataframe.

In [1]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print(response.status_code)

200


In [1]:
data = pd.json_normalize(response.json())
data.head()

   flight_number                  date_utc     rocket     payloads    launchpad                                                                                                                                                     cores
0              1  2010-06-04T00:00:00.000Z  rocket_f9  [payload_0]  launchpad_0      [{'core': 'core_0', 'flight': 1, 'gridfins': False, 'reused': False, 'legs': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]
1              2  2012-05-22T00:00:00.000Z  rocket_f9  [payload_1]  launchpad_0      [{'core': 'core_1', 'flight': 1, 'gridfins': False, 'reused': False, 'legs': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]
2              3  2013-03-01T00:00:00.000Z  rocket_f9  [payload_2]  launchpad_0      [{'core': 'core_2', 'flight': 1, 'gridfins': False, 'reused': False, 'legs': False, 'landing_success': None, 'landing_type': None, 'landpad': None}]
3              4  2013-09-29T00:00:00.000Z  rocket_f9  [payload_

We take a subset of the columns we need, and filter out launches with more than one core or more than one payload (these are Falcon Heavy / rideshare missions handled separately).

In [1]:
data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]

data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]

data['cores'] = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])

data['date'] = pd.to_datetime(data['date_utc']).dt.date

data = data[data['date'] <= datetime.date(2020, 11, 13)]
print(f"{data.shape[0]} launches remain after filtering to single-core, single-payload, pre-cutoff-date missions")

91 launches remain after filtering to single-core, single-payload, pre-cutoff-date missions


### Extract booster version, launch site, payload, and core data

Call the helper functions defined above to populate the parallel lists.

In [1]:
BoosterVersion = []
getBoosterVersion(data)
BoosterVersion[0:5]

['Falcon 9', 'Falcon 9', 'Falcon 9', 'Falcon 9', 'Falcon 9']

In [1]:
LaunchSite = []
Longitude = []
Latitude = []
getLaunchSite(data)

In [1]:
PayloadMass = []
Orbit = []
getPayloadData(data)

In [1]:
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
getCoreData(data)

### Assemble the launch dictionary and construct the dataframe

In [1]:
launch_dict = {
    'FlightNumber': list(data['flight_number']),
    'Date': list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude,
}

launch_df = pd.DataFrame(launch_dict)
launch_df.head()

   FlightNumber        Date BoosterVersion  PayloadMass Orbit    LaunchSite      Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  ReusedCount Serial   Longitude   Latitude
0             1  2010-06-04       Falcon 9  6104.959412   LEO  CCAFS SLC 40    None None        1     False   False  False       None    1.0            0  B0003  -80.577366  28.561857
1             2  2012-05-22       Falcon 9   525.000000   LEO  CCAFS SLC 40    None None        1     False   False  False       None    1.0            0  B0005  -80.577366  28.561857
2             3  2013-03-01       Falcon 9   677.000000   ISS  CCAFS SLC 40    None None        1     False   False  False       None    1.0            0  B0007  -80.577366  28.561857
3             4  2013-09-29       Falcon 9   500.000000    PO   VAFB SLC 4E  False Ocean        1     False   False  False       None    1.0            0  B1003 -120.610829  34.632093
4             5  2013-12-03       Falcon 9  3170.000000   GTO  CCAFS SLC 40    N

### Filter the dataframe to Falcon 9 launches only

Remove any Falcon 1 launches and renumber `FlightNumber` sequentially.

In [1]:
data_falcon9 = launch_df[launch_df['BoosterVersion'] != 'Falcon 1'].copy()
data_falcon9.loc[:, 'FlightNumber'] = list(range(1, data_falcon9.shape[0] + 1))
print(f"{launch_df.shape[0]} total launches -> {data_falcon9.shape[0]} Falcon 9 launches after removing Falcon 1 missions")
data_falcon9.head()

91 total launches -> 90 Falcon 9 launches after removing Falcon 1 missions


   FlightNumber        Date BoosterVersion  PayloadMass Orbit    LaunchSite      Outcome  Flights  GridFins  Reused   Legs LandingPad  Block  ReusedCount Serial   Longitude   Latitude
0             1  2010-06-04       Falcon 9  6104.959412   LEO  CCAFS SLC 40    None None        1     False   False  False       None    1.0            0  B0003  -80.577366  28.561857
1             2  2012-05-22       Falcon 9   525.000000   LEO  CCAFS SLC 40    None None        1     False   False  False       None    1.0            0  B0005  -80.577366  28.561857
2             3  2013-03-01       Falcon 9   677.000000   ISS  CCAFS SLC 40    None None        1     False   False  False       None    1.0            0  B0007  -80.577366  28.561857
3             4  2013-09-29       Falcon 9   500.000000    PO   VAFB SLC 4E  False Ocean        1     False   False  False       None    1.0            0  B1003 -120.610829  34.632093
4             5  2013-12-03       Falcon 9  3170.000000   GTO  CCAFS SLC 40    N

### Deal with missing values

Check which columns have missing values.

In [1]:
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0

`PayloadMass` has some missing values. We replace them with the column's mean, a standard, defensible imputation strategy for a roughly-symmetric numeric feature. `LandingPad` legitimately has missing values (no landing was attempted), so those are left as-is -- they are meaningful, not noise.

In [1]:
payload_mean = data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass'] = data_falcon9['PayloadMass'].fillna(payload_mean)
data_falcon9.isnull().sum()

FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0

In [1]:
data_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Saved dataset_part_1.csv with", data_falcon9.shape[0], "rows and", data_falcon9.shape[1], "columns")

Saved dataset_part_1.csv with 90 rows and 17 columns


## Conclusion

We collected 94 past launch records from the SpaceX API, filtered out multi-core/multi-payload
missions and launches after the 2020-11-13 cutoff, removed the single Falcon 1 launch, and imputed
the small number of missing `PayloadMass` values with the column mean. The resulting
`dataset_part_1.csv` (90 Falcon 9 launches) is the foundation for the data-wrangling, EDA, and
predictive-modeling notebooks that follow.